# Nepal: Tiger Recovery & Forest Cover Change (2000–2025)

This notebook analyzes two intertwined conservation stories in Nepal over the past 25 years:

1. **Tiger population recovery** — Nepal's national tiger census counts (2000–2026)
2. **Forest cover change** — national forest extent from government assessments and global remote-sensing datasets (Hansen/UMD Global Forest Change, hosted on Global Forest Watch)

**Why these two together?** Nepal's tigers live almost entirely within the Terai Arc Landscape (Chitwan, Bardiya, Parsa, Banke, and Shuklaphanta National Parks and connecting forest corridors). Tiger recovery is generally credited to anti-poaching enforcement, protected-area management, and **community forestry**, which has also helped stabilize national forest cover. This notebook explores whether the data show that relationship, and is careful not to overstate causality.

> **A note on data access:** this environment has no live internet access, so this notebook uses **real, cited figures** from official census reports and forest assessments (compiled below) rather than fabricated numbers. Section 5 gives ready-to-run code templates for pulling live raster data (Hansen/GFW, OpenStreetMap boundaries) that you can run on your own machine or in Google Colab, where internet access is available.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 1. Tiger population data

Nepal has run a national tiger census roughly every four years since 2009, using camera-trap surveys across the Terai Arc Landscape. Figures below are drawn from Nepal's Department of National Parks and Wildlife Conservation (DNPWC) census releases, reported via the Kathmandu Post, IUCN, and Tiger Encounter / Tx2 tracking pages.

| Year | National tiger count | Source / notes |
|---|---|---|
| 2000 | 109 (estimate) | Pre-camera-trap estimate |
| 2005 | 126 (estimate) | Pre-camera-trap estimate |
| 2009 | 121 | First scientific camera-trap census (baseline for Tx2 doubling goal) |
| 2013 | 198 | Second census |
| 2018 | 235 | Third census |
| 2022 | 355 | Fourth census — surpassed Tx2 goal of 250 |
| 2026 | 429 | Fifth census, announced July 29, 2026 (World Tiger Day) |

The 2000/2005 figures predate systematic camera-trap methodology and are widely cited estimates rather than census counts — they are kept for long-run context but shown with a dashed line to flag the methodology change at 2009.


In [ ]:
tiger_data = pd.DataFrame({
    "year": [2000, 2005, 2009, 2013, 2018, 2022, 2026],
    "tiger_count": [109, 126, 121, 198, 235, 355, 429],
    "method": ["estimate", "estimate", "camera_trap", "camera_trap",
               "camera_trap", "camera_trap", "camera_trap"],
})
tiger_data["pct_change_from_2009"] = (
    (tiger_data["tiger_count"] / 121 - 1) * 100
).round(1)
tiger_data


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

estimate_mask = tiger_data["method"] == "estimate"
census_mask = ~estimate_mask

# Dashed segment for pre-2009 estimates, solid for the scientific census era
ax.plot(tiger_data.loc[tiger_data["year"] <= 2009, "year"],
        tiger_data.loc[tiger_data["year"] <= 2009, "tiger_count"],
        "--o", color="#b35806", label="Pre-2009 estimate")
ax.plot(tiger_data.loc[tiger_data["year"] >= 2009, "year"],
        tiger_data.loc[tiger_data["year"] >= 2009, "tiger_count"],
        "-o", color="#1b7837", linewidth=2.5, label="Camera-trap census")

for _, row in tiger_data.iterrows():
    ax.annotate(int(row["tiger_count"]), (row["year"], row["tiger_count"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)

ax.axhline(250, color="gray", linestyle=":", linewidth=1)
ax.text(2000.5, 258, "Tx2 goal: 250 tigers by 2022", fontsize=8, color="gray")

ax.set_title("Nepal's Wild Tiger Population, 2000–2026")
ax.set_xlabel("Year")
ax.set_ylabel("Estimated / counted tigers")
ax.legend()
plt.tight_layout()
plt.show()

growth_2009_2026 = (429 / 121 - 1) * 100
print(f"Growth from 2009 baseline to 2026: +{growth_2009_2026:.0f}%  "
      f"(121 -> 429 tigers, a {429/121:.1f}x increase)")


## 2. Forest cover data

Forest cover assessments in Nepal come from several methodologies that are **not directly comparable** (aerial photo inventories, Landsat-based land cover mapping, and high-resolution RapidEye-based Forest Resource Assessments), so figures below are kept with their source and method noted rather than smoothed into one series.

| Year | Forest cover (% of land area) | Source / method |
|---|---|---|
| ~1994 | 39.6% | National Forest Inventory (aerial photo / field plots) |
| 2010 | 39.1% | Uddin et al. 2015, Landsat-based national land cover map |
| 2010–2014 | 44.74% | Dept. of Forest Research & Survey, Forest Resource Assessment (FRA), RapidEye 5m imagery — forest + other wooded land |
| 2020 | ~44% | Global Forest Watch / UMD, natural forest tree cover (Hansen-derived, 30m Landsat) |
| 2021 | 40.36% | National Forest Inventory 2021 |

Despite methodological noise, multiple independent studies agree on the broad trend: **Nepal's forest cover was roughly stable-to-increasing from the 1990s through the 2020s**, a widely cited conservation success attributed largely to community forestry (over 22,000 Community Forest User Groups now manage close to 2.9 million hectares).

Separately, annual **tree cover loss** (disturbance, which can include harvesting followed by regrowth, not necessarily permanent deforestation) is tracked continuously by Global Forest Watch using the Hansen/UMD Global Forest Change dataset. In 2020 Nepal had about 6.5 million hectares of natural forest (44% of land area); in 2025 it lost roughly 4,900 hectares of natural forest, a small fraction of the total — illustrating that recent annual loss is minor relative to standing forest stock.


In [ ]:
forest_data = pd.DataFrame({
    "year": [1994, 2010, 2012, 2020, 2021],
    "forest_pct": [39.6, 39.1, 44.74, 44.0, 40.36],
    "source": ["NFI (aerial/field)", "Uddin et al. 2015 (Landsat)",
               "DFRS FRA 2010-2014 (RapidEye 5m)",
               "Global Forest Watch / UMD (Landsat 30m)",
               "National Forest Inventory 2021"],
})
forest_data


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {"NFI (aerial/field)": "#2166ac", "Uddin et al. 2015 (Landsat)": "#2166ac",
          "DFRS FRA 2010-2014 (RapidEye 5m)": "#b2182b",
          "Global Forest Watch / UMD (Landsat 30m)": "#1b7837",
          "National Forest Inventory 2021": "#2166ac"}

for src in forest_data["source"].unique():
    sub = forest_data[forest_data["source"] == src]
    ax.scatter(sub["year"], sub["forest_pct"], s=90, color=colors[src], label=src, zorder=3)

ax.plot(forest_data["year"], forest_data["forest_pct"], color="lightgray",
        linestyle="--", zorder=1)

ax.set_title("Nepal Forest Cover Estimates by Source, 1994–2021")
ax.set_xlabel("Year")
ax.set_ylabel("Forest cover (% of land area)")
ax.set_ylim(30, 50)
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

print("Note: points come from different methodologies and are not a single continuous")
print("time series -- compare within-source trends, not point-to-point across sources.")


## 3. Bringing the two together

Tiger habitat in Nepal is concentrated in the Terai Arc, so it's natural to ask whether forest cover trends track tiger population trends. The chart below overlays both series on a shared timeline (dual y-axis). Because the two datasets don't share the exact same measurement years, values are lightly interpolated **only for visualization** — interpolated points are marked and should not be read as measured data.


In [ ]:
years_common = np.arange(2000, 2027)

tiger_interp = np.interp(years_common, tiger_data["year"], tiger_data["tiger_count"])
forest_interp = np.interp(years_common, forest_data["year"], forest_data["forest_pct"])

fig, ax1 = plt.subplots(figsize=(9, 5))
ax2 = ax1.twinx()

ax1.plot(years_common, tiger_interp, color="#1b7837", linewidth=2, label="Tiger count (interpolated)")
ax1.scatter(tiger_data["year"], tiger_data["tiger_count"], color="#1b7837", zorder=3, s=40,
            label="Tiger count (measured)")

ax2.plot(years_common, forest_interp, color="#b2182b", linewidth=2, linestyle="--",
         label="Forest cover % (interpolated)")
ax2.scatter(forest_data["year"], forest_data["forest_pct"], color="#b2182b", zorder=3,
            marker="s", s=40, label="Forest cover % (measured)")

ax1.set_xlabel("Year")
ax1.set_ylabel("Tiger count", color="#1b7837")
ax2.set_ylabel("Forest cover (%)", color="#b2182b")
ax1.tick_params(axis="y", labelcolor="#1b7837")
ax2.tick_params(axis="y", labelcolor="#b2182b")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper left")

ax1.set_title("Nepal: Tiger Population vs. Forest Cover, 2000–2026")
plt.tight_layout()
plt.show()

corr = np.corrcoef(tiger_interp, forest_interp)[0, 1]
print(f"Correlation between interpolated tiger count and forest cover %: r = {corr:.2f}")
print("Interpretation: interpolation makes any correlation partly an artifact of the")
print("shared timeline. Treat this as a descriptive illustration, not a causal test --")
print("tiger recovery is attributed in the literature mainly to anti-poaching enforcement")
print("and protected-area management, with community forestry supporting habitat quality")
print("rather than forest cover alone driving tiger numbers.")


## 4. Key findings

- **Tigers**: Nepal's wild tiger population rose from a 2009 camera-trap baseline of 121 to 429 in 2026 — a **3.5x increase in 17 years**, comfortably exceeding the international Tx2 commitment to double tiger numbers by 2022.
- **By park (2026 vs. 2022)**: gains in Chitwan (128→145), Parsa (41→71), Banke (25→51), and Shuklaphanta (36→50); Bardiya was the only park to decline (125→112).
- **Forests**: national forest cover estimates cluster around **39–45%** of land area across 1994–2021, depending on method — broadly stable to slightly increasing rather than the steep decline seen in many tropical forest nations over the same period.
- **Recent disturbance**: Global Forest Watch recorded roughly 4,900 hectares of natural forest loss in Nepal in 2025, small relative to the ~6.5 million hectares of natural forest present in 2020.
- **Community forestry** (22,000+ Community Forest User Groups managing ~2.9 million hectares) is the most frequently cited driver of Nepal's forest cover resilience, alongside protected-area expansion that has also benefited tiger habitat.


## 5. Optional: pulling live geospatial data (requires internet)

This sandbox has no network access, so the datasets above are compiled from published reports rather than downloaded live. If you run this notebook somewhere with internet access (e.g. Google Colab, or your own machine), the templates below show how to pull the actual underlying raster/vector layers used by most Nepal forest-change studies:

- **Nepal boundary** via OpenStreetMap (`osmnx` / Nominatim)
- **Hansen Global Forest Change** (2000–2024 tree cover, loss, gain) via Google Earth Engine, clipped to Nepal
- **Protected area boundaries** (Chitwan, Bardiya, Parsa, Banke, Shuklaphanta) via the World Database on Protected Areas (WDPA)

These cells are provided as templates (not executed here) — install the packages and authenticate as needed.


In [ ]:
# --- Template only: requires internet + Earth Engine authentication ---
# pip install earthengine-api geemap osmnx --break-system-packages

# import ee, geemap, osmnx as ox
#
# ee.Authenticate()
# ee.Initialize()
#
# # 1) Nepal boundary from OpenStreetMap
# nepal = ox.geocode_to_gdf("Nepal")
# nepal_geom = ee.Geometry(nepal.geometry.iloc[0].__geo_interface__)
#
# # 2) Hansen Global Forest Change dataset, clipped to Nepal
# gfc = ee.Image("UMD/hansen/global_forest_change_2024_v1_12").clip(nepal_geom)
# tree_cover_2000 = gfc.select("treecover2000")
# loss_year = gfc.select("lossyear")       # pixel value = year of loss (1-24 -> 2001-2024)
# gain = gfc.select("gain")                # cumulative gain 2000-2012
#
# # 3) Example: total tree-cover-loss area (ha) by year, Nepal-wide
# pixel_area = ee.Image.pixelArea().divide(10000)  # m^2 -> ha
# for yr in range(1, 25):
#     loss_mask = loss_year.eq(yr).And(tree_cover_2000.gt(30))
#     loss_ha = pixel_area.updateMask(loss_mask).reduceRegion(
#         reducer=ee.Reducer.sum(), geometry=nepal_geom, scale=30, maxPixels=1e13
#     ).get("area")
#     print(2000 + yr, ee.Number(loss_ha).getInfo())
#
# # 4) Protected areas (Terai Arc parks) from WDPA, for spatial overlay with tiger census zones
# wdpa = ee.FeatureCollection("WCMC/WDPA/current/polygons") \
#     .filter(ee.Filter.eq("ISO3", "NPL"))


## 6. Sources

- Nepal DNPWC national tiger census results, reported in Kathmandu Post (2022, 2026), IUCN, The Tribune, and Tiger Encounter / Tx2 tracking pages
- Global Forest Watch, Nepal country dashboard (globalforestwatch.org/dashboards/country/NPL)
- Global Forest Change dataset (Hansen et al. 2013; Potapov et al. 2022), University of Maryland / Google Earth Engine
- Department of Forest Research and Survey (DFRS), Forest Resource Assessment 2010–2014, Nepal
- Uddin, K. et al. 2015, "Development of 2010 national land cover database for Nepal"
- National Forest Inventory 2021, Government of Nepal, Ministry of Forests and Environment
- "Monitoring Forest Cover Trends in Nepal: Insights from 2000–2020," *Sustainability*, 2025 (mdpi.com/2071-1050/17/14/6511)

*Compiled July 2026. Figures are drawn from the cited public reports; always verify against the primary source before use in formal publications.*
